# Pitcher Data
This notebook is intended to create the `pitchers.csv` file that the main project uses.

In [1]:

import time
import pandas as pd
from pybaseball import pitching_stats_bref, chadwick_register, cache

cache.enable()  # caches downloads so reruns are fast

Get every pitcher season in the range 2016-2026

In [2]:
START, END = 2016, 2026

frames = []
for season in range(START, END + 1):
    try:
        df = pitching_stats_bref(season)
        df = df[["mlbID", "Name"]].copy()
        df["season"] = season
        frames.append(df)
        print(f"{season}: {len(df)} pitchers")
    except Exception as e:
        print(f"{season}: failed -> {e}")
    time.sleep(3)  # be polite to Baseball Reference

seasons = pd.concat(frames, ignore_index=True)
seasons["mlbID"] = pd.to_numeric(seasons["mlbID"], errors="coerce")
seasons = seasons.dropna(subset=["mlbID"]).astype({"mlbID": int})

# one row per pitcher, with the seasons they pitched in the window
window = (
    seasons.groupby("mlbID")
    .agg(
        window_first_season=("season", "min"),
        window_last_season=("season", "max"),
        seasons_pitched=("season", "nunique"),
    )
    .reset_index()
)
print(len(window), "unique pitchers")

2016: 742 pitchers
2017: 755 pitchers
2018: 799 pitchers
2019: 831 pitchers
2020: 735 pitchers
2021: 909 pitchers
2022: 871 pitchers
2023: 863 pitchers
2024: 855 pitchers
2025: 873 pitchers
2026: 869 pitchers
2626 unique pitchers


Join to the Chadwick Reigster (register of all professional baseball players)

In [3]:
reg = chadwick_register()

reg = reg[
    [
        "key_mlbam", "key_bbref", "key_fangraphs", "key_retro",
        "name_first", "name_last",
        "mlb_played_first", "mlb_played_last",
    ]
].copy()
reg["key_mlbam"] = pd.to_numeric(reg["key_mlbam"], errors="coerce")
reg = reg.dropna(subset=["key_mlbam"]).astype({"key_mlbam": int})

pitchers = window.merge(reg, left_on="mlbID", right_on="key_mlbam", how="left")

pitchers["name"] = (pitchers["name_first"].fillna("") + " " + pitchers["name_last"].fillna("")).str.strip()

pitchers = pitchers.rename(columns={
    "mlbID": "mlbam_id",
    "key_bbref": "bbref_id",
    "key_fangraphs": "fangraphs_id",
    "key_retro": "retrosheet_id",
    "mlb_played_first": "career_start",
    "mlb_played_last": "career_end",
})

# career_end is only meaningful if the player has retired:
# if their last season is the latest one in the data, treat them as active (blank)
pitchers.loc[pitchers["career_end"] >= END, "career_end"] = pd.NA

# fall back to the window seasons where the register has no dates
pitchers["career_start"] = pitchers["career_start"].fillna(pitchers["window_first_season"])

cols = [
    "mlbam_id", "bbref_id", "fangraphs_id", "retrosheet_id", "name",
    "career_start", "career_end",
    "window_first_season", "window_last_season", "seasons_pitched",
]
pitchers = pitchers[cols].sort_values("name").reset_index(drop=True)
pitchers.to_csv("pitchers.csv", index=False)

print(pitchers.shape)
pitchers.head(10)

(2626, 10)


,mlbam_id,bbref_id,fangraphs_id,retrosheet_id,name,career_start,career_end,window_first_season,window_last_season,seasons_pitched
0,592091,achteaj01,11387,achta001,A. J. Achter,2014.0,2016.0,2016,2016,1
1,669935,alexyaj01,22343,alexa001,A. J. Alexy,2021.0,2022.0,2021,2022,2
2,595918,coleaj01,11467,colea002,A. J. Cole,2015.0,2021.0,2016,2021,6
3,456167,griffaj01,11132,grifa002,A. J. Griffin,2012.0,2017.0,2016,2017,2
4,607609,ladwiaj01,16266,ladwa001,A. J. Ladwig,2022.0,2022.0,2022,2022,1
5,621345,minteaj01,18655,minta001,A. J. Minter,2017.0,NaN,2017,2026,10
6,571963,morriaj01,9919,morra002,A. J. Morris,2016.0,2016.0,2016,2016,1
7,640462,pukaj01,19343,puk-a001,A. J. Puk,2019.0,2025.0,2019,2025,6
8,607223,reedaj01,16246,reeda002,A. J. Reed,2016.0,2019.0,2019,2019,1
9,519263,schugaj01,11432,schua001,A. J. Schugel,2015.0,2017.0,2016,2017,2


Sanity Check

In [4]:
print("duplicate ids:", pitchers["mlbam_id"].duplicated().sum())
print("missing names:", (pitchers["name"] == "").sum())
print("missing bbref ids:", pitchers["bbref_id"].isna().sum())
print("active (no career_end):", pitchers["career_end"].isna().sum())


duplicate ids: 0
missing names: 0
missing bbref ids: 0
active (no career_end): 920


In [5]:
import numpy as np
import pandas as pd
from pybaseball import (
    pitching_stats, playerid_reverse_lookup, cache,
    statcast_pitcher_arsenal_stats,
    statcast_pitcher_expected_stats,
    statcast_pitcher_exitvelo_barrels,
)

cache.enable()

# ---------------- Config ----------------
START, END = 2015, 2024      # seasons to aggregate over
MIN_TBF = 200                # minimum total batters faced (FanGraphs path)
MIN_PA = 100                 # minimum total PA (Statcast fallback path)
OUT = "pitcher_features.csv"

# ---------------- Helpers ----------------
def to_pct_scale(s):
    s = pd.to_numeric(s.astype(str).str.replace("%", "", regex=False), errors="coerce")
    return s * 100 if s.dropna().max() <= 1.5 else s

def wmean(g, col, w):
    m = g[col].notna() & (g[w] > 0)
    return np.average(g.loc[m, col], weights=g.loc[m, w]) if m.any() else np.nan

def slope(g, col):
    x = g[["Season", col]].dropna()
    return np.polyfit(x["Season"], x[col], 1)[0] if len(x) >= 3 else np.nan

# ---------------- Path A: FanGraphs ----------------
FG_RATES = [
    "K%", "BB%", "K-BB%", "HR/9", "HR/FB", "GB%", "FB%", "LD%", "IFFB%",
    "O-Swing%", "Z-Swing%", "O-Contact%", "Z-Contact%", "Zone%",
    "F-Strike%", "SwStr%", "CSW%",
    "FA% (pi)", "SL% (pi)", "CT% (pi)", "CB% (pi)", "CH% (pi)", "SF% (pi)", "KN% (pi)",
    "FBv", "SLv", "CBv", "CHv",
    "Pull%", "Oppo%", "Soft%", "Hard%",
    "Stuff+", "Location+", "Pitching+", "Age",
]

def fangraphs_features():
    frames = []
    for yr in range(START, END + 1):
        d = pitching_stats(yr, yr, qual=1)
        if d is None or d.empty or "Name" not in d.columns:
            raise RuntimeError(f"FanGraphs returned no usable data for {yr}")
        d["Season"] = yr
        frames.append(d)
        print("FanGraphs", yr, d.shape)
    raw = pd.concat(frames, ignore_index=True)

    present = [c for c in FG_RATES if c in raw.columns]
    print("FanGraphs columns not found:", sorted(set(FG_RATES) - set(present)))
    for c in present:
        if c.endswith("%") or c.endswith("% (pi)"):
            raw[c] = to_pct_scale(raw[c])

    rows = []
    for fgid, g in raw.groupby("IDfg"):
        if g["TBF"].sum() < MIN_TBF:
            continue
        G = max(g["G"].sum(), 1)
        r = {"IDfg": fgid, "name": g.sort_values("Season")["Name"].iloc[-1]}
        r.update({c: wmean(g, c, "TBF") for c in present})
        r.update({
            "first_season": g["Season"].min(), "last_season": g["Season"].max(),
            "seasons_pitched": g["Season"].nunique(),
            "IP_total": g["IP"].sum(), "TBF_total": g["TBF"].sum(),
            "GS_share": g["GS"].sum() / G, "IP_per_G": g["IP"].sum() / G,
            "SV_per_G": g["SV"].sum() / G,
        })
        for c in ("FBv", "K%", "BB%"):
            if c in g.columns:
                r[f"{c}_slope"] = slope(g, c)
        rows.append(r)
    feats = pd.DataFrame(rows)

    # Attach MLBAM ids (optional, for joining to Statcast later)
    try:
        ids = playerid_reverse_lookup(feats["IDfg"].astype(int).tolist(), key_type="fangraphs")
        ids = ids[["key_fangraphs", "key_mlbam"]].rename(
            columns={"key_fangraphs": "IDfg", "key_mlbam": "mlbam_id"})
        feats = feats.merge(ids.drop_duplicates("IDfg"), on="IDfg", how="left")
    except Exception as e:
        print("MLBAM lookup skipped:", repr(e))
    return feats

# ---------------- Path B: Baseball Savant fallback ----------------
def savant_name(df):
    if "last_name, first_name" in df.columns:
        parts = df["last_name, first_name"].str.split(",", n=1, expand=True)
        return (parts[1].str.strip() + " " + parts[0].str.strip())
    return df["first_name"].str.strip() + " " + df["last_name"].str.strip()

def savant_features():
    seasons = []
    for yr in range(max(START, 2015), END + 1):
        try:
            ars = statcast_pitcher_arsenal_stats(yr, minPA=25)
            exp = statcast_pitcher_expected_stats(yr, minPA=25)
            ev = statcast_pitcher_exitvelo_barrels(yr, minBBE=25)
        except Exception as e:
            print("Savant", yr, "FAILED:", repr(e))
            continue

        wide = ars.pivot_table(index="player_id", columns="pitch_type",
                               values=["pitch_usage", "whiff_percent"], aggfunc="first")
        wide.columns = [f"{a}_{b}" for a, b in wide.columns]
        wide = wide.reset_index()

        exp = exp.copy()
        exp["name"] = savant_name(exp)
        exp = exp[["player_id", "name", "pa", "est_woba", "est_ba", "est_slg", "xera"]]

        ev_cols = [c for c in ["player_id", "avg_hit_speed", "brl_percent",
                               "ev95percent", "anglesweetspotpercent"] if c in ev.columns]
        d = exp.merge(ev[ev_cols], on="player_id", how="left").merge(wide, on="player_id", how="left")
        d["Season"] = yr
        seasons.append(d)
        print("Savant", yr, d.shape)

    if not seasons:
        raise RuntimeError("Both FanGraphs and Savant failed. Upgrade pybaseball and check your connection.")
    raw = pd.concat(seasons, ignore_index=True)
    num = [c for c in raw.columns if c not in ("player_id", "name", "pa", "Season")]

    rows = []
    for pid, g in raw.groupby("player_id"):
        if g["pa"].sum() < MIN_PA:
            continue
        r = {"mlbam_id": pid, "name": g.sort_values("Season")["name"].iloc[-1]}
        r.update({c: wmean(g, c, "pa") for c in num})
        r.update({"first_season": g["Season"].min(), "last_season": g["Season"].max(),
                  "seasons_pitched": g["Season"].nunique(), "PA_total": g["pa"].sum(),
                  "est_woba_slope": slope(g, "est_woba")})
        rows.append(r)
    return pd.DataFrame(rows)

# ---------------- Run ----------------
try:
    features = fangraphs_features()
    source = "FanGraphs"
except Exception as e:
    print("\nFanGraphs path failed:", repr(e))
    print("Falling back to Baseball Savant leaderboards...\n")
    features = savant_features()
    source = "Savant"

features = features.sort_values("name").reset_index(drop=True)
features.to_csv(OUT, index=False)
print(f"\nSource: {source} | saved {OUT} | shape {features.shape}")
print(features[["name"]].head())
print(features.isna().mean().sort_values(ascending=False).head(15))


FanGraphs path failed: RuntimeError('FanGraphs returned no usable data for 2015')
Falling back to Baseball Savant leaderboards...

Savant 2015 (735, 12)
Savant 2016 (742, 12)
Savant 2017 (755, 34)
Savant 2018 (799, 32)
Savant 2019 (831, 32)
Savant 2020 (735, 30)
Savant 2021 (909, 30)
Savant 2022 (871, 30)
Savant 2023 (863, 34)
Savant 2024 (855, 34)

Source: Savant | saved pitcher_features.csv | shape (1643, 39)
           name
0   A.J. Achter
1    A.J. Alexy
2  A.J. Burnett
3     A.J. Cole
4  A.J. Griffin
whiff_percent_SC    0.999391
pitch_usage_SC      0.999391
pitch_usage_FO      0.998174
whiff_percent_KN    0.998174
pitch_usage_KN      0.998174
whiff_percent_FO    0.998174
whiff_percent_SV    0.991479
pitch_usage_SV      0.991479
whiff_percent_FS    0.927572
pitch_usage_FS      0.927572
pitch_usage_ST      0.855143
whiff_percent_ST    0.855143
whiff_percent_FC    0.757152
pitch_usage_FC      0.757152
pitch_usage_CU      0.650030
dtype: float64
